# Working Example to Highlight New CSM usage

This example is very basic, and will be used to demonstrate inputs and outputs as the model is developed.

In [1]:
from attrs import fields

from csm.base_model import CSMBase

The `CSMBase` is where all the base data validation and inputs will be provided. This
class will also store key information to be able to automatically generate
WISDEM-compliant models (a possibly working implementation exists at `csm/wisdem_api.py`).

In [2]:
# an input
blade_mass_coeff = fields(CSMBase).blade_mass_coeff
print(f"Blade mass coefficient default value: {blade_mass_coeff.default}")
print(f"Blade mass coefficient metadata: {blade_mass_coeff.metadata}")
print()

# an output that can also be provided as input
blade_mass = fields(CSMBase).blade_mass
print(f"Blade mass default value: {blade_mass.default}")
print(f"Blade mass metadata: {blade_mass.metadata}")

Blade mass coefficient default value: None
Blade mass coefficient metadata: {'units': 'unitless', 'io': 'input'}

Blade mass default value: None
Blade mass metadata: {'units': 'kg', 'io': 'both'}


## Creating a model from the base model

This is a simple demonstration of how to create a model. Please note this is in progress, in
particular, there is not yet an implementation of the 2015 model, so the defaults that will get
applied to the base model in a subclass are kept separate from the sample inputs for tracking
purposes.

In [3]:
# default values from past implementations
defaults_2015_land = {
    "efficiency_max": 1.0,
    "blade_mass_coeff": 0.5,
    "blade_mass_cost_coeff": 14.6,
    "hub_mass_coeff": 2.3,
    "hub_mass_intercept": 1320.0,
    "hub_mass_cost_coeff": 3.9,
}

# example input
csm_2015_inputs = {
    "rated_power_kw": 3500,
    "turbine_class": 1,
    "blade_has_carbon": False,
    "rotor_diameter": 136.7,
    "max_tip_speed": 63.0,
}

rep15 = CSMBase(**(defaults_2015_land | csm_2015_inputs))
rep15.run()

## IRS Tables

**Notes**
1. Only blade and hub calculations have been fully implemented, so everything else is
   placeholder values (e.g., nacelle cost) or nonexistent (e.g., tower cost).
2. The inputs to `irs_mpc_breakdown` incorporate inputs that are not included in CSM
   equations, but could be added as model inputs if desired.

In [4]:
rep15.irs_mpc_breakdown(
    turbine_production_cost=1500,
    tower_flange_material_cost=2000,
    tower_flange_production_cost=1000,
)

cost      Value
APC                                  MPC                                      
Wind Turbine                         Blade            333855.625193   60.63278
                                     Hub              210263.408081  38.186731
                                     Nacelle                 1000.0   0.181614
                                     Power Converter         1000.0   0.181614
                                     Production              1500.0   0.272421
Wind Tower Flange                    Material                2000.0   0.363228
                                     Production              1000.0   0.181614
Tower                                -                            -          -
Steel or iron products in foundation -                            -          -
Total                                -                550619.033274      100.0

In [5]:
def total_domestic_content(
    cls,
    turbine_production_cost: float,
    tower_flange_material_cost: float,
    tower_flange_production_cost: float,
    domestic: list[str],
    *,
    return_table: bool = False,
) -> float:
    breakdown = cls.irs_mpc_breakdown(
        turbine_production_cost=turbine_production_cost,
        tower_flange_material_cost=tower_flange_material_cost,
        tower_flange_production_cost=tower_flange_production_cost,
        with_category=True,
    )
    domestic_map = dict.fromkeys(breakdown.category, 0)
    domestic_map |= dict.fromkeys(domestic, 1)
    breakdown = breakdown.assign(Domestic=breakdown.category.map(domestic_map))

    for apc in ("Wind Turbine", "Wind Tower Flange"):
        component = breakdown.loc[apc, "Domestic"]
        if component.size != component.sum():
            breakdown.loc[apc, "Domestic"] = 0.0

    breakdown.Domestic = breakdown.Domestic * (breakdown.Value / 100)
    breakdown.loc["Total", "Domestic"] = breakdown.Domestic.sum()
    breakdown.Domestic *= 100
    if return_table:
        return breakdown
    return breakdown.loc["Total", "Domestic"].squeeze()


domestic_mpc = total_domestic_content(
    rep15,
    turbine_production_cost=1500,
    tower_flange_material_cost=2000,
    tower_flange_production_cost=1000,
    domestic=["tower_flange_material", "tower_flange_production", "nacelle"],
)
print(f"Total Domestic content: {domestic_mpc:.2f}%")

Total Domestic content: 0.54%


In [6]:
total_domestic_content(
    rep15,
    turbine_production_cost=1500,
    tower_flange_material_cost=2000,
    tower_flange_production_cost=1000,
    domestic=["tower_flange_material", "tower_flange_production", "nacelle"],
    return_table=True,
)

category  \
APC                                  MPC                                        
Wind Turbine                         Blade                              blade   
                                     Hub                                  hub   
                                     Nacelle                          nacelle   
                                     Power Converter          power_converter   
                                     Production            turbine_production   
Wind Tower Flange                    Material           tower_flange_material   
                                     Production       tower_flange_production   
Tower                                -                                      -   
Steel or iron products in foundation -                                      -   
Total                                -                                      -   

                                                               cost  \
APC                                  MPC                              
Wind Turbine                         Blade            333855.625193   
                                     Hub              210263.408081   
                                     Nacelle                 1000.0   
                                     Power Converter         1000.0   
                                     Production              1500.0   
Wind Tower Flange                    Material                2000.0   
                                     Production              1000.0   
Tower                                -                          0.0   
Steel or iron products in foundation -                          0.0   
Total                                -                550619.033274   

                                                          Value  Domestic  
APC                                  MPC                                   
Wind Turbine                         Blade             60.63278       0.0  
                                     Hub              38.186731       0.0  
                                     Nacelle           0.181614       0.0  
                                     Power Converter   0.181614       0.0  
                                     Production        0.272421       0.0  
Wind Tower Flange                    Material          0.363228  0.363228  
                                     Production        0.181614  0.181614  
Tower                                -                      0.0       0.0  
Steel or iron products in foundation -                      0.0       0.0  
Total                                -                    100.0  0.544841